# Creating an FVCOM Restart File Using CMEMS Data

This tutorial demonstrates how to create an FVCOM restart file using ocean data from the Copernicus Marine Environment Monitoring Service (CMEMS). The process involves reading CMEMS reanalysis data, interpolating it onto the FVCOM grid, and writing it out in the proper restart file format.

## Overview

FVCOM (Finite Volume Community Ocean Model) restart files contain the initial conditions needed to start or restart a model simulation. These files include variables such as:
- Sea surface height (`zeta`)
- Temperature (`temp`) 
- Salinity (`salinity`)
- Velocity components (`u`, `v`)

CMEMS provides high-quality ocean reanalysis data that can be used to initialize FVCOM models with realistic oceanographic conditions.

## Prerequisites

Before running this tutorial, ensure you have:
- Access to CMEMS data files (both 2D and 3D products)
- A donor FVCOM restart file (template with correct grid structure)
- The PyFVCOM2 package installed

## Step 1: Import Required Libraries

First, we'll import all the necessary modules from PyFVCOM2 and standard Python libraries:

In [ ]:
import pathlib
from datetime import datetime
import numpy as np

from pyfvcom2.file_utils import find_file
from pyfvcom2.cmems_reader import CMEMSReader
from pyfvcom2.fvcom_reader import FVCOMReader
from pyfvcom2.interpolation import InterpolationCoordinates, CMEMSInterpolator
from pyfvcom2.restart import write_restart

## Step 2: Configure Paths and Parameters

Next, we'll set up the file paths and parameters needed for the interpolation process:

- `cmems_data_dir`: Root directory containing CMEMS data
- `product_name_2d/3d`: Specific CMEMS product names for 2D and 3D variables
- `donor_filepath`: Path to an existing FVCOM restart file that provides the target grid
- `start_date_time`: The date/time for which we want to create initial conditions
- `fvcom_to_cmems_var_names`: Mapping between FVCOM and CMEMS variable names

In [ ]:
# Paths etc
cmems_data_dir = '/data/sthenno1/scratch/jcl/wcoof/data/CMEMS_rolling'
product_name_2d = 'cmems_mod_nws_phy_anfc_0.027deg-2D_PT1H-m'
product_name_3d = 'cmems_mod_nws_phy_anfc_0.027deg-3D_PT1H-m'
donor_filepath = '/users/modellers/jcl/Projects/wcoof_cylc_rosesuite/wcoof_suite/etc/tamar_v2_donor_restart.nc'
start_date_time = datetime.strptime("20250614", "%Y%m%d")

# FVCOM -> CMEMS name map. This is the same as the default name map, but we include it here to demonstrate how it
# can be used to define equivalent variables names in FVCOM and CMEMS data.
fvcom_to_cmems_var_names = {'temp': 'thetao',
                            'salinity': 'so',
                            'u': 'uo',
                            'v': 'vo',
                            'zeta': 'zos'}

## Step 3: Set Up Interpolation Coordinates

Before interpolating the data, we need to create interpolation coordinate objects that specify where on the FVCOM grid we want the data to be interpolated. Different variables are located at different positions:

- **Node variables** (like temperature, salinity, sea surface height): Located at the vertices of the triangular mesh
- **Element variables** (like velocity components): Located at the centers of the triangular elements

We'll get the spatial coordinates from the FVCOM reader and add the temporal information:

In [ ]:
# FVCOM reader for the restart
fvcom_reader = FVCOMReader(donor_filepath)

# Interpolation coordinates for variables defined at nodes
interp_coords_nodes = fvcom_reader.get_interpolation_coordinates('node')
interp_coords_nodes.dates = np.asarray([start_date_time])

# Interpolation coordinates for variables defined at elements
interp_coords_elements = fvcom_reader.get_interpolation_coordinates('element')
interp_coords_elements.dates = np.asarray([start_date_time])

## Step 4: Process 2D Variables

Now we'll handle the 2D variables (variables that don't vary with depth). In this case, we're processing sea surface height:

1. **Find the CMEMS data file**: Use the `find_file` utility to locate the correct file containing data for our target date
2. **Create CMEMS reader**: Initialize a reader for the 2D data file
3. **Create interpolator**: Set up a CMEMS interpolator with variable name mapping
4. **Interpolate**: Use the interpolator to map CMEMS data onto the FVCOM grid

In [ ]:
# 2D data
# -------
cmems_data_dir_2d = pathlib.Path(cmems_data_dir, product_name_2d)
file_path, time_index = find_file(cmems_data_dir_2d, product_name_2d, start_date_time)

# Create reader
cmems_reader_2d = CMEMSReader(file_path, reference_var_name='zos')

# Create interpolator (2D)
interpolator = CMEMSInterpolator(cmems_reader_2d, fvcom_to_cmems_var_names)

# 2D vars
var_data_2d = {}
var_data_2d['zeta'] = interpolator.interpolate(interp_coords_nodes, 'zeta')

## Step 5: Process 3D Variables

Now we'll handle the 3D variables (variables that vary with depth). These include temperature, salinity, and velocity components:

- **Temperature** (`'thetao'` → `'temp'`): Located at model nodes
- **Salinity** (`'so'` → `'salinity'`): Located at model nodes  
- **U-velocity** (`'uo'` → `'u'`): Located at element centers
- **V-velocity** (`'vo'` → `'v'`): Located at element centers

The process is similar to 2D variables but uses the 3D CMEMS product and handles multiple variables:

In [ ]:
# 3D data 
# -------     
cmems_data_dir_3d = pathlib.Path(cmems_data_dir, product_name_3d)
file_path, time_index = find_file(cmems_data_dir_3d, product_name_3d, start_date_time)

# Create reader
cmems_reader_3d = CMEMSReader(file_path, reference_var_name='so')

# Create interpolator (3D)
interpolator = CMEMSInterpolator(cmems_reader_3d, fvcom_to_cmems_var_names)

# 3D vars
vars_3d = {'u': 'element',
           'v': 'element',
           'temp': 'node',
           'salinity': 'node'}

var_data_3d = {}
for var_name, position in vars_3d.items():
    if position == 'node':
        var_data_3d[var_name] = interpolator.interpolate(interp_coords_nodes, var_name)
    else: # 'element'
        var_data_3d[var_name] = interpolator.interpolate(interp_coords_elements, var_name)

## Step 6: Combine Results

We'll combine the 2D and 3D variable dictionaries to create our complete dataset:

In [ ]:
# Combine data
var_data = var_data_2d | var_data_3d

## Step 7: Write the Restart File

Now we'll create the actual FVCOM restart file using the interpolated data. The `write_restart` function takes:

- The donor file path (provides the grid structure and metadata)
- The output file path 
- The dictionary of interpolated variables
- The start date/time for the restart

In [ ]:
# Create the restart
# ------------------

# Create data directory for the restart
restart_dir = pathlib.Path('data/restart')
restart_dir.mkdir(parents=True, exist_ok=True)

restart_file = pathlib.Path(f'{restart_dir}', 'tamar_cmems_restart_test.nc')
write_restart(donor_filepath, restart_file, var_data, start_date_time)

## Step 8: Visualize the Results

Finally, let's create a simple visualization to verify that our restart file was created correctly. We'll plot the surface temperature field using PyFVCOM2's plotting capabilities:

In [ ]:
# Plot the result
# ---------------
from matplotlib import pyplot as plt
import cartopy.crs as ccrs
from netCDF4 import Dataset

from pyfvcom2.plotting import FVCOMPlotter, create_figure

In [ ]:
# Create an FVCOM plotter for our restart file
plotter = FVCOMPlotter(restart_file)

# Create a figure with geographic projection
fig, ax = create_figure(projection=ccrs.PlateCarree())

# Open the restart file and extract surface temperature
ds = Dataset(restart_file)
temp = ds.variables['temp']

# Plot the surface temperature (last sigma level, time index 0)
plotter.plot_field(ax, temp[0, 0, :])

plt.show()

## Summary

Congratulations! You have successfully created an FVCOM restart file using CMEMS reanalysis data. Here's what we accomplished:

1. **Data Integration**: Combined high-quality oceanographic data from CMEMS with FVCOM's unstructured grid format
2. **Spatial Interpolation**: Converted data from CMEMS's regular grid to FVCOM's triangular mesh using the new `CMEMSInterpolator` class
3. **Variable Mapping**: Correctly mapped between different variable naming conventions using the `fvcom_to_cmems_var_names` dictionary
4. **File Creation**: Generated a properly formatted NetCDF restart file ready for use in FVCOM

## Key Changes in the New API

The updated tutorial uses PyFVCOM2's new interpolation architecture:

- **InterpolationCoordinates**: Unified coordinate handling for different grid positions
- **CMEMSInterpolator**: Dedicated interpolator class for CMEMS data with automatic 2D/3D handling
- **Direct interpolation**: More explicit control over the interpolation process
- **Variable name mapping**: Flexible mapping between FVCOM and CMEMS variable names

This new approach provides better separation of concerns and more flexibility for different data sources and interpolation methods.